In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml

sys.path.append("/workspace")

from vertexcbf.config_utils import build_constr_fn, build_dynamics
from vertexcbf.mpc import beam_search, branch_and_bound, stochastic_beam_search, mppi, cem, icem, random_shooting, cem_discrete

%reload_ext autoreload
%autoreload 2

WORKSPACE = "/workspace"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


In [ ]:
SYSTEM = "vertical_drone_2d"

WORKSPACE = "/workspace"
CONFIG_PATH = f"{WORKSPACE}/configs/{SYSTEM}.yaml"

# 2-D slice configuration: only the slice axes are gridded; the remaining
# state dims are held at the values given in SLICE_FIXED_VALUES.
# SLICE_SHAPE is ignored when `{TRUE_VALUES_DIR}/grid.npy` exists — the slice
# is taken from that ground-truth grid instead.
SLICE_SHAPE = (100, 100)   # resolution of the slice grid (n0, n1)
SLICE_AXES = (0, 1)        # state dims plotted on the x and y axis
SLICE_AXES_LABELS = (r"$\theta$", r"$\omega$")  # labels for the slice axes (for plotting)
SLICE_FIXED_VALUES = {}    # {axis_index: value} for non-slice dims (nx > 2)


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

#==============================================================================

## Load config, dynamics, and trained model
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

dynamics = build_dynamics(cfg["system"], device=device)
constr_fn = build_constr_fn(cfg["constraint"])

#==============================================================================

## 2-D slice of the value function
"""
Build the grid only over `SLICE_AXES`, holding the remaining state dims at `SLICE_FIXED_VALUES`,
then run the closed-loop validation rollout on those exact slice states. The figure overlays:
"""

x_min = dynamics.x_min.cpu().numpy()
x_max = dynamics.x_max.cpu().numpy()

ax0, ax1 = SLICE_AXES
x_label, y_label = SLICE_AXES_LABELS

other_axes = [i for i in range(dynamics.nx) if i not in (ax0, ax1)]


missing = [i for i in other_axes if i not in SLICE_FIXED_VALUES]
if missing:
    raise ValueError(
        f"SLICE_FIXED_VALUES must specify values for axes {missing}; "
        f"got keys {sorted(SLICE_FIXED_VALUES)}"
    )

n0, n1 = SLICE_SHAPE
x0 = np.linspace(x_min[ax0], x_max[ax0], n0)
x1 = np.linspace(x_min[ax1], x_max[ax1], n1)
g0, g1 = np.meshgrid(x0, x1, indexing="ij")

slice_states = np.zeros((n0 * n1, dynamics.nx), dtype=np.float32)
slice_states[:, ax0] = g0.ravel()
slice_states[:, ax1] = g1.ravel()
for i in other_axes:
    slice_states[:, i] = SLICE_FIXED_VALUES[i]

slice_states_t = torch.as_tensor(slice_states, dtype=torch.float32, device=device)

result_mpc = beam_search(
    dynamics=dynamics,
    constr_fn=constr_fn,
    x0=slice_states_t,
    B=500,
    K=20,
    dt=0.1,
)

# result_mpc = stochastic_beam_search(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=1000,
#     K=100,
#     dt=0.02,
#     strategy="gumbel_topk", # 
#     temperature=0.05,
# )

# result_mpc = branch_and_bound(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=1000,
#     K=60,
#     dt=0.05,
#     n_restarts=2,
# )

# result_mpc = mppi(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     N_s=2000,
#     K=60,
#     dt=0.05,
#     sigma=0.1,
#     lam=0.0,
#     n_iter=5,
# )

values_mpc = result_mpc["values"].cpu().numpy().reshape(n0, n1)

constr = constr_fn(slice_states_t).cpu().numpy().reshape(n0, n1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(g0, g1, values_mpc, cmap="plasma", shading="auto")

ax.contour(g0, g1, constr, levels=[0],
           colors="black", linewidths=2.0, linestyles="-")

    
ax.contour(g0, g1, values_mpc, levels=[0],
           colors="dodgerblue", linewidths=1.5, linestyles="--")


cbar = plt.colorbar(im, ax=ax, location="right")
cbar.ax.set_title(r"$V_{\Theta}(x)$")

In [ ]:
SYSTEM = "quadruped_trunk"

WORKSPACE = "/workspace"
CONFIG_PATH = f"{WORKSPACE}/configs/{SYSTEM}.yaml"

# 2-D slice configuration: only the slice axes are gridded; the remaining
# state dims are held at the values given in SLICE_FIXED_VALUES.
# SLICE_SHAPE is ignored when `{TRUE_VALUES_DIR}/grid.npy` exists — the slice
# is taken from that ground-truth grid instead.
SLICE_SHAPE = (100, 100)   # resolution of the slice grid (n0, n1)
SLICE_AXES = (1, 2)        # state dims plotted on the x and y axis
SLICE_AXES_LABELS = (r"$\theta$", r"$q$")  # labels for the slice axes (for plotting)
SLICE_FIXED_VALUES = {
    0: 0.3,
    # 1: 0.0,
    # 2: 0.0,
    3: 0.0,
    4: 0.0,
    5: 0.0,
    6: 0.0,
    7: 0.0,
    8: 0.0,
}    # {axis_index: value} for non-slice dims (nx > 2)


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

#==============================================================================

## Load config, dynamics, and trained model
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

dynamics = build_dynamics(cfg["system"], device=device)
constr_fn = build_constr_fn(cfg["constraint"])

#==============================================================================

## 2-D slice of the value function
"""
Build the grid only over `SLICE_AXES`, holding the remaining state dims at `SLICE_FIXED_VALUES`,
then run the closed-loop validation rollout on those exact slice states. The figure overlays:
"""

x_min = dynamics.x_min.cpu().numpy()
x_max = dynamics.x_max.cpu().numpy()

ax0, ax1 = SLICE_AXES
x_label, y_label = SLICE_AXES_LABELS

other_axes = [i for i in range(dynamics.nx) if i not in (ax0, ax1)]


missing = [i for i in other_axes if i not in SLICE_FIXED_VALUES]
if missing:
    raise ValueError(
        f"SLICE_FIXED_VALUES must specify values for axes {missing}; "
        f"got keys {sorted(SLICE_FIXED_VALUES)}"
    )

n0, n1 = SLICE_SHAPE
x0 = np.linspace(x_min[ax0], x_max[ax0], n0)
x1 = np.linspace(x_min[ax1], x_max[ax1], n1)
g0, g1 = np.meshgrid(x0, x1, indexing="ij")

slice_states = np.zeros((n0 * n1, dynamics.nx), dtype=np.float32)
slice_states[:, ax0] = g0.ravel()
slice_states[:, ax1] = g1.ravel()
for i in other_axes:
    slice_states[:, i] = SLICE_FIXED_VALUES[i]

slice_states_t = torch.as_tensor(slice_states, dtype=torch.float32, device=device)

# result_mpc = beam_search(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=800,
#     K=100,
#     dt=0.01,
# )

# result_mpc = stochastic_beam_search(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=1000,
#     K=100,
#     dt=0.02,
#     strategy="gumbel_topk", # 
#     temperature=0.05,
# )

result_mpc = mppi(
    dynamics=dynamics,
    constr_fn=constr_fn,
    x0=slice_states_t,
    N_s=1000,
    K=100,
    dt=0.02,
    sigma=2.2,
    lam=0.0,
    n_iter=5,
)

# result_mpc = branch_and_bound(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=800,
#     K=0,
#     dt=0.05,
#     n_restarts=1,
# )

values_mpc = result_mpc["values"].cpu().numpy().reshape(n0, n1)

constr = constr_fn(slice_states_t).cpu().numpy().reshape(n0, n1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(g0, g1, values_mpc, cmap="plasma", shading="auto")

ax.contour(g0, g1, constr, levels=[0],
           colors="black", linewidths=2.0, linestyles="-")

    
ax.contour(g0, g1, values_mpc, levels=[0],
           colors="dodgerblue", linewidths=1.5, linestyles="--")


cbar = plt.colorbar(im, ax=ax, location="right")
cbar.ax.set_title(r"$V_{\Theta}(x)$")

In [ ]:
SYSTEM = "auv_6dof"

WORKSPACE = "/workspace"
CONFIG_PATH = f"{WORKSPACE}/configs/{SYSTEM}.yaml"

# 2-D slice configuration: only the slice axes are gridded; the remaining
# state dims are held at the values given in SLICE_FIXED_VALUES.
# SLICE_SHAPE is ignored when `{TRUE_VALUES_DIR}/grid.npy` exists — the slice
# is taken from that ground-truth grid instead.
SLICE_SHAPE = (100, 100)   # resolution of the slice grid (n0, n1)
SLICE_AXES = (1, 2)        # state dims plotted on the x and y axis
SLICE_AXES_LABELS = (r"$p_z$", r"$w$")  # labels for the slice axes (for plotting)
SLICE_FIXED_VALUES = {
    0: 0.0,
    # 1: 0.0,
    # 2: 0.0,
    3: 0.0,
    4: 0.0,
    5: 1.0,
    6: 1.6,
    7: 0.0,
    8: -1.6,
    9: 0.0,
    10: 0.0,
    11: 0.0,
}    # {axis_index: value} for non-slice dims (nx > 2)


device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

#==============================================================================

## Load config, dynamics, and trained model
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

dynamics = build_dynamics(cfg["system"], device=device)
constr_fn = build_constr_fn(cfg["constraint"])

#==============================================================================

## 2-D slice of the value function
"""
Build the grid only over `SLICE_AXES`, holding the remaining state dims at `SLICE_FIXED_VALUES`,
then run the closed-loop validation rollout on those exact slice states. The figure overlays:
"""

x_min = dynamics.x_min.cpu().numpy()
x_max = dynamics.x_max.cpu().numpy()

ax0, ax1 = SLICE_AXES
x_label, y_label = SLICE_AXES_LABELS

other_axes = [i for i in range(dynamics.nx) if i not in (ax0, ax1)]


missing = [i for i in other_axes if i not in SLICE_FIXED_VALUES]
if missing:
    raise ValueError(
        f"SLICE_FIXED_VALUES must specify values for axes {missing}; "
        f"got keys {sorted(SLICE_FIXED_VALUES)}"
    )

n0, n1 = SLICE_SHAPE
x0 = np.linspace(x_min[ax0], x_max[ax0], n0)
x1 = np.linspace(x_min[ax1], x_max[ax1], n1)
g0, g1 = np.meshgrid(x0, x1, indexing="ij")

slice_states = np.zeros((n0 * n1, dynamics.nx), dtype=np.float32)
slice_states[:, ax0] = g0.ravel()
slice_states[:, ax1] = g1.ravel()
for i in other_axes:
    slice_states[:, i] = SLICE_FIXED_VALUES[i]

slice_states_t = torch.as_tensor(slice_states, dtype=torch.float32, device=device)

# result_mpc = beam_search(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=600,
#     K=50,
#     dt=0.1,
# )

result_mpc = stochastic_beam_search(
    dynamics=dynamics,
    constr_fn=constr_fn,
    x0=slice_states_t,
    B=800,
    K=40,
    dt=0.1,
    strategy="gumbel_topk", # 
    temperature=0.08,
)

# result_mpc = branch_and_bound(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=800,
#     K=40,
#     dt=0.1,
#     n_restarts=2,
# )

values_mpc = result_mpc["values"].cpu().numpy().reshape(n0, n1)

constr = constr_fn(slice_states_t).cpu().numpy().reshape(n0, n1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(g0, g1, values_mpc, cmap="plasma", shading="auto")

ax.contour(g0, g1, constr, levels=[0],
           colors="black", linewidths=2.0, linestyles="-")

    
ax.contour(g0, g1, values_mpc, levels=[0],
           colors="dodgerblue", linewidths=1.5, linestyles="--")


cbar = plt.colorbar(im, ax=ax, location="right")
cbar.ax.set_title(r"$V_{\Theta}(x)$")

In [ ]:
SYSTEM = "double_integrator_1d"

WORKSPACE = "/workspace"
CONFIG_PATH = f"{WORKSPACE}/configs/{SYSTEM}.yaml"

# 2-D slice configuration: only the slice axes are gridded; the remaining
# state dims are held at the values given in SLICE_FIXED_VALUES.
# SLICE_SHAPE is ignored when `{TRUE_VALUES_DIR}/grid.npy` exists — the slice
# is taken from that ground-truth grid instead.
SLICE_SHAPE = (100, 100)   # resolution of the slice grid (n0, n1)
SLICE_AXES = (0, 1)        # state dims plotted on the x and y axis
SLICE_AXES_LABELS = (r"$p$", r"$v$")  # labels for the slice axes (for plotting)
SLICE_FIXED_VALUES = None


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

#==============================================================================

## Load config, dynamics, and trained model
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

dynamics = build_dynamics(cfg["system"], device=device)
constr_fn = build_constr_fn(cfg["constraint"])

#==============================================================================

## 2-D slice of the value function
"""
Build the grid only over `SLICE_AXES`, holding the remaining state dims at `SLICE_FIXED_VALUES`,
then run the closed-loop validation rollout on those exact slice states. The figure overlays:
"""

x_min = dynamics.x_min.cpu().numpy()
x_max = dynamics.x_max.cpu().numpy()

ax0, ax1 = SLICE_AXES
x_label, y_label = SLICE_AXES_LABELS

other_axes = [i for i in range(dynamics.nx) if i not in (ax0, ax1)]


missing = [i for i in other_axes if i not in SLICE_FIXED_VALUES]
if missing:
    raise ValueError(
        f"SLICE_FIXED_VALUES must specify values for axes {missing}; "
        f"got keys {sorted(SLICE_FIXED_VALUES)}"
    )

n0, n1 = SLICE_SHAPE
x0 = np.linspace(x_min[ax0], x_max[ax0], n0)
x1 = np.linspace(x_min[ax1], x_max[ax1], n1)
g0, g1 = np.meshgrid(x0, x1, indexing="ij")

slice_states = np.zeros((n0 * n1, dynamics.nx), dtype=np.float32)
slice_states[:, ax0] = g0.ravel()
slice_states[:, ax1] = g1.ravel()
for i in other_axes:
    slice_states[:, i] = SLICE_FIXED_VALUES[i]

slice_states_t = torch.as_tensor(slice_states, dtype=torch.float32, device=device)

# result_mpc = beam_search(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=2000,
#     K=40,
#     dt=0.1,
# )

result_mpc = stochastic_beam_search(
    dynamics=dynamics,
    constr_fn=constr_fn,
    x0=slice_states_t,
    B=2000,
    K=40,
    dt=0.1,
    strategy="gumbel_topk", # 
    temperature=0.05,
)

# result_mpc = branch_and_bound(
#     dynamics=dynamics,
#     constr_fn=constr_fn,
#     x0=slice_states_t,
#     B=1500,
#     K=80,
#     dt=0.05,
#     n_restarts=2,
# )

values_mpc = result_mpc["values"].cpu().numpy().reshape(n0, n1)

constr = constr_fn(slice_states_t).cpu().numpy().reshape(n0, n1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.pcolormesh(g0, g1, values_mpc, cmap="plasma", shading="auto")

ax.contour(g0, g1, constr, levels=[0],
           colors="black", linewidths=2.0, linestyles="-")

    
ax.contour(g0, g1, values_mpc, levels=[0],
           colors="dodgerblue", linewidths=1.5, linestyles="--")


cbar = plt.colorbar(im, ax=ax, location="right")
cbar.ax.set_title(r"$V_{\Theta}(x)$")